# 🔍 Analitik Big Data — Final Project
## Analisis Pasar Kerja Data Science Indonesia via JobStreet
---
**Mata Kuliah:** Analitik Big Data  
**Kelas:** TIF-C  
**Dataset:** `raw_dataset_jobstreet.csv` — lowongan kerja Data Science dari JobStreet Indonesia  

### Pipeline Overview
```
Cell 1  → Environment Setup         (install PySpark, Java, libraries)
Cell 2  → SparkSession Init          (konfigurasi Spark)
Cell 3  → Data Ingestion & Schema    (load & inspect data)
Cell 4  → Cleaning & Preprocessing  (normalisasi data)
Cell 5  → EDA via Spark SQL          (query analitik)
Cell 6  → Visualization              (charts & wordcloud)
Cell 7  → MLlib K-Means Clustering   (NLP + clustering)
Cell 8  → Cluster Interpretation     (wordcloud per cluster)
Cell 9  → Summary & Cleanup          (ringkasan pipeline)
```

In [1]:
# Environment Setup
# Menginstal dan mengkonfigurasi semua dependensi yang diperlukan:
#   - OpenJDK 17  : Java runtime yang dibutuhkan PySpark
#   - PySpark      : Framework komputasi big data berbasis Apache Spark
#   - matplotlib   : Library visualisasi dasar
#   - seaborn      : Library visualisasi statistik berbasis matplotlib
#   - wordcloud    : Library untuk membuat visualisasi word cloud
# Setelah instalasi, variabel lingkungan JAVA_HOME dikonfigurasi
# agar PySpark dapat menemukan instalasi Java.

import subprocess, sys

print("[1/4] Menginstal OpenJDK 17...")
subprocess.run(
    ["apt-get", "install", "-y", "-q", "openjdk-17-jdk"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

print("[2/4] Menginstal PySpark...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.1"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

print("[3/4] Menginstal matplotlib, seaborn, wordcloud...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "matplotlib", "seaborn", "wordcloud"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

print(" [4/4] Mengkonfigurasi JAVA_HOME...")
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# Verifikasi instalasi Java
java_ver = subprocess.run(["java", "-version"],
                          capture_output=True, text=True).stderr.split("\n")[0]
print(f"\n Lingkungan siap!")
print(f"    Java  : {java_ver}")
print(f"    Python: {sys.version.split()[0]}")
print(f"    JAVA_HOME: {os.environ['JAVA_HOME']}")

[1/4] Menginstal OpenJDK 17...
[2/4] Menginstal PySpark...
[3/4] Menginstal matplotlib, seaborn, wordcloud...
 [4/4] Mengkonfigurasi JAVA_HOME...

 Lingkungan siap!
    Java  : openjdk version "17.0.19" 2026-04-21
    Python: 3.12.13
    JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


In [3]:
# SparkSession Initialization
# Menginisialisasi SparkSession sebagai entry point utama untuk semua
# operasi PySpark dalam notebook ini.
#
# Konfigurasi:
#   - appName        : "JobMarket_BigData_Analytics"
#   - master         : local[*]  → menggunakan semua core CPU yang tersedia
#   - driver.memory  : 2g        → 2 GB memory untuk driver node
#   - log level      : ERROR     → hanya tampilkan error (bukan INFO/WARN)

from pyspark.sql import SparkSession

print("Menginisialisasi SparkSession...")

spark = (
    SparkSession.builder
    .appName("JobMarket_BigData_Analytics")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print(f"\nSparkSession berhasil diinisialisasi!")
print(f"    Spark Version  : {spark.version}")
print(f"    Master        : {spark.sparkContext.master}")
print(f"    App Name       : {spark.sparkContext.appName}")
print(f"    Default Parallelism: {spark.sparkContext.defaultParallelism}")

Menginisialisasi SparkSession...

SparkSession berhasil diinisialisasi!
    Spark Version  : 3.5.1
    Master        : local[*]
    App Name       : JobMarket_BigData_Analytics
    Default Parallelism: 2


In [6]:
# Data Ingestion & Schema Inspection
# Memuat dataset CSV mentah (raw) ke dalam PySpark DataFrame.
#
# Opsi pembacaan:
#   - header=True       : baris pertama sebagai nama kolom
#   - inferSchema=True  : otomatis mendeteksi tipe data tiap kolom
#   - multiLine=True    : mendukung teks multiline dalam kolom
#   - escape='"'        : escape karakter untuk tanda kutip ganda

import os

# Path dataset
CSV_PATH = "raw_dataset_jobstreet.csv"

# Cek keberadaan file
if not os.path.exists(CSV_PATH):
    # Coba path alternatif di Colab
    alt_paths = [
        f"/content/{CSV_PATH}",
        f"/content/drive/MyDrive/{CSV_PATH}"
    ]
    for p in alt_paths:
        if os.path.exists(p):
            CSV_PATH = p
            break
    else:
        raise FileNotFoundError(
            f"File '{CSV_PATH}' tidak ditemukan.\n"
            "   Upload file ke Colab terlebih dahulu dengan:\n"
            "   from google.colab import files; files.upload()"
        )

print(f"Memuat dataset dari: {CSV_PATH}")

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .option("quote", '"')
    .csv(CSV_PATH)
)

# Simpan jumlah baris awal untuk ringkasan akhir
total_raw = df_raw.count()

print(f"\nSchema DataFrame:")
df_raw.printSchema()

print(f"\n Total Baris Data  : {total_raw:,} baris")
print(f"   Total Kolom       : {len(df_raw.columns)} kolom")
print(f"   Kolom             : {df_raw.columns}")

print(f"\n Preview 5 Baris Pertama:")
df_raw.show(5, truncate=80)

Memuat dataset dari: raw_dataset_jobstreet.csv

Schema DataFrame:
root
 |-- job_title: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- job_type: string (nullable = true)
 |-- experience_level: string (nullable = true)
 |-- education_req: string (nullable = true)
 |-- salary_range: string (nullable = true)
 |-- job_requirements: string (nullable = true)
 |-- job_responsibilities: string (nullable = true)
 |-- posted_date: string (nullable = true)
 |-- source_platform: string (nullable = true)


 Total Baris Data  : 150 baris
   Total Kolom       : 11 kolom
   Kolom             : ['job_title', 'company_name', 'location', 'job_type', 'experience_level', 'education_req', 'salary_range', 'job_requirements', 'job_responsibilities', 'posted_date', 'source_platform']

 Preview 5 Baris Pertama:
+-------------------------------------------------------+-------------------------------+------------+----------------+------------------

In [8]:
# Data Cleaning & Preprocessing

# Melakukan serangkaian proses pembersihan dan normalisasi data menggunakan
# PySpark DataFrame API.

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

print("Memulai proses data cleaning...")

# LANGKAH 0: Hitung null count SEBELUM cleaning (sebagai baseline)
print("\n Menghitung null count SEBELUM cleaning...")

def get_null_counts(df, label="DataFrame"):
    """Hitung jumlah null per kolom dan kembalikan sebagai dict."""
    null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c)
                  for c in df.columns]
    return df.agg(*null_exprs).collect()[0].asDict()

null_before = get_null_counts(df_raw, "SEBELUM")

# LANGKAH 1: Ganti "Tidak tersedia" → null
print("\n [1/4] Mengganti 'Tidak tersedia' dengan null...")

TARGET_TEXT  = "Tidak tersedia"
SKIP_COLUMNS = {"source_platform"}

df_clean = df_raw
for col_name in df_raw.columns:
    if col_name not in SKIP_COLUMNS:
        df_clean = df_clean.withColumn(
            col_name,
            F.when(
                F.col(col_name).isNotNull() &
                (F.trim(F.col(col_name)) == TARGET_TEXT),
                F.lit(None)
            ).otherwise(F.col(col_name))
        )

print(f"    Selesai — 'Tidak tersedia' → null pada {len(df_raw.columns) - len(SKIP_COLUMNS)} kolom")

# LANGKAH 2: Normalisasi job_type
print("\n [2/4] Normalisasi kolom job_type...")

# Pola pencocokan (case-insensitive) berdasarkan kata kunci dalam kolom job_type
df_clean = df_clean.withColumn(
    "job_type",
    F.when(F.col("job_type").isNull(), F.lit(None).cast("string"))
     .when(F.lower(F.col("job_type")).rlike(r"full.?time|penuh waktu"), F.lit("Full time"))
     .when(F.lower(F.col("job_type")).rlike(r"part.?time|paruh waktu"), F.lit("Part time"))
     .when(F.lower(F.col("job_type")).rlike(r"kontrak|contract|temporer"), F.lit("Kontrak"))
     .when(F.lower(F.col("job_type")).rlike(r"intern|magang"), F.lit("Internship/Magang"))
     .when(F.lower(F.col("job_type")).rlike(r"freelance|kasual|casual"), F.lit("Freelance"))
     .otherwise(F.lit("Lainnya"))
)

job_type_dist = df_clean.groupBy("job_type").count().orderBy(F.desc("count"))
print("   Distribusi job_type setelah normalisasi:")
job_type_dist.show(truncate=False)

# LANGKAH 3: Normalisasi salary_range → salary_numeric
print(" [3/4] Normalisasi salary_range → salary_numeric (dalam ribuan Rupiah)...")

# Ekstrak angka pertama dari salary_range:
#   Format contoh: "Rp 3.300.000 – Rp 3.800.000 per month"
#   Langkah: hapus "Rp", hapus titik ribuan, ambil angka pertama
df_clean = df_clean.withColumn(
    "salary_cleaned_tmp",
    F.regexp_replace(F.col("salary_range"), r"[Rp\s,]", "")
).withColumn(
    "salary_cleaned_tmp",
    F.regexp_replace(F.col("salary_cleaned_tmp"), r"\.", "")
).withColumn(
    "salary_numeric",
    F.when(
        F.col("salary_range").isNull(),
        F.lit(None).cast(DoubleType())
    ).otherwise(
        F.regexp_extract(
            F.col("salary_cleaned_tmp"),
            r"(\d+)",
            1
        ).cast(DoubleType())
    )
).drop("salary_cleaned_tmp")

# Tampilkan contoh hasil
print("   Contoh hasil salary_numeric (5 baris dengan salary non-null):")
df_clean.filter(F.col("salary_numeric").isNotNull()) \
        .select("salary_range", "salary_numeric") \
        .show(5, truncate=60)

# LANGKAH 4: Normalisasi posted_date → posted_days_ago
print(" [4/4] Normalisasi posted_date → posted_days_ago (hari)...")

# Format yang dijumpai di dataset:
#   "9 jam yang lalu"    → 0 hari
#   "2 hari yang lalu"   → 2 hari
#   "30+ hari yang lalu" → 30 hari
#   "1 hari yang lalu"   → 1 hari
df_clean = df_clean.withColumn(
    "posted_days_ago",
    F.when(
        F.col("posted_date").isNull(),
        F.lit(None).cast(IntegerType())
    ).when(
        F.lower(F.col("posted_date")).rlike(r"jam"),
        F.lit(0).cast(IntegerType())          # jam yang lalu → 0 hari
    ).when(
        F.lower(F.col("posted_date")).rlike(r"hari"),
        F.regexp_extract(
            F.col("posted_date"), r"(\d+)\+?", 1
        ).cast(IntegerType())                 # N hari yang lalu → N
    ).when(
        F.lower(F.col("posted_date")).rlike(r"minggu"),
        (F.regexp_extract(
            F.col("posted_date"), r"(\d+)", 1
        ).cast(IntegerType()) * 7)            # N minggu → N*7 hari
    ).when(
        F.lower(F.col("posted_date")).rlike(r"bulan"),
        (F.regexp_extract(
            F.col("posted_date"), r"(\d+)", 1
        ).cast(IntegerType()) * 30)           # N bulan → N*30 hari
    ).otherwise(
        F.lit(None).cast(IntegerType())
    )
)

print("   Contoh hasil posted_days_ago (10 baris):")
df_clean.select("posted_date", "posted_days_ago").show(10, truncate=40)

# LANGKAH 5: Perbandingan null count SEBELUM vs SESUDAH
print("\n Perbandingan Null Count: SEBELUM vs SESUDAH Cleaning")
print()

null_after = get_null_counts(df_clean, "SESUDAH")

print(f"{'Kolom':<30} {'Sebelum':>8} {'Sesudah':>8} {'Berkurang':>10}")
print("-" * 60)
for col_name in df_raw.columns:
    before = null_before.get(col_name, 0)
    after  = null_after.get(col_name, 0)
    diff   = before - after
    print(f"{col_name:<30} {before:>8} {after:>8} {diff:>10}")

# Kolom baru yang ditambahkan
for new_col in ["salary_numeric", "posted_days_ago"]:
    after = null_after.get(new_col, 0)
    print(f"{new_col:<30} {'(baru)':>8} {after:>8} {'N/A':>10}")
print("=" * 65)

# LANGKAH 6: Simpan hasil cleaning
OUTPUT_CLEANED = "cleaned_dataset_jobstreet.csv"
print(f"\n Menyimpan dataset yang sudah dibersihkan ke '{OUTPUT_CLEANED}'...")

(
    df_clean
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(OUTPUT_CLEANED)
)

total_clean = df_clean.count()
print(f" Dataset berhasil disimpan!")
print(f"    Total baris setelah cleaning : {total_clean:,}")
print(f"    Lokasi output               : {OUTPUT_CLEANED}/")

Memulai proses data cleaning...

 Menghitung null count SEBELUM cleaning...

 [1/4] Mengganti 'Tidak tersedia' dengan null...
    Selesai — 'Tidak tersedia' → null pada 10 kolom

 [2/4] Normalisasi kolom job_type...
   Distribusi job_type setelah normalisasi:
+---------+-----+
|job_type |count|
+---------+-----+
|Full time|115  |
|Kontrak  |30   |
|Freelance|5    |
+---------+-----+

 [3/4] Normalisasi salary_range → salary_numeric (dalam ribuan Rupiah)...
   Contoh hasil salary_numeric (5 baris dengan salary non-null):
+---------------------------------------+--------------+
|                           salary_range|salary_numeric|
+---------------------------------------+--------------+
|                    Rp 500,000 per hour|      500000.0|
|  Rp 3.300.000 – Rp 3.800.000 per month|     3300000.0|
| Rp 7.000.000 – Rp 10.000.000 per month|     7000000.0|
|Rp 10.000.000 – Rp 15.000.000 per month|         1.0E7|
|  Rp 3.000.000 – Rp 4.000.000 per month|     3000000.0|
+-----------------

In [9]:
# Exploratory Data Analysis (EDA) via Spark SQL

# Melakukan analisis eksploratif menggunakan Spark SQL.
# DataFrame yang sudah dibersihkan didaftarkan sebagai temp view "job_market"
# sehingga dapat diquery menggunakan sintaks SQL standar.
#
# Analisis yang dilakukan:
#   Q1. Top 10 kota dengan lowongan terbanyak
#   Q2. Distribusi job_type (jumlah + persentase)
#   Q3. Top 10 perusahaan dengan lowongan terbanyak
#   Q4. Distribusi waktu posting (1 hari, 2-7 hari, 8-30 hari, >30 hari)
#   Q5. Rata-rata salary_numeric per kota (top 10, exclude null salary)

from pyspark.sql import functions as F

# Daftarkan DataFrame sebagai Spark SQL temporary view
df_clean.createOrReplaceTempView("job_market")
print(" Temp view 'job_market' berhasil dibuat")
print(f"   Total baris dalam view: {spark.sql('SELECT COUNT(*) FROM job_market').collect()[0][0]:,}\n")

# Q1: Top 10 Kota dengan Lowongan Terbanyak
print(" Q1: Top 10 Kota dengan Lowongan Terbanyak")

q1 = spark.sql("""
    SELECT
        TRIM(location)   AS kota,
        COUNT(*)         AS jumlah_lowongan
    FROM job_market
    WHERE location IS NOT NULL
    GROUP BY TRIM(location)
    ORDER BY jumlah_lowongan DESC
    LIMIT 10
""")
q1.show(truncate=50)

# Q2: Distribusi Job Type (Jumlah + Persentase)
print(" Q2: Distribusi Job Type")

q2 = spark.sql("""
    SELECT
        COALESCE(job_type, 'Tidak Diketahui') AS tipe_pekerjaan,
        COUNT(*) AS jumlah,
        ROUND(
            COUNT(*) * 100.0 / (SELECT COUNT(*) FROM job_market),
            2
        ) AS persentase
    FROM job_market
    GROUP BY job_type
    ORDER BY jumlah DESC
""")
q2.show(truncate=50)

# Q3: Top 10 Perusahaan dengan Lowongan Terbanyak
print(" Q3: Top 10 Perusahaan dengan Lowongan Terbanyak")

q3 = spark.sql("""
    SELECT
        TRIM(company_name) AS perusahaan,
        COUNT(*)           AS jumlah_lowongan
    FROM job_market
    WHERE company_name IS NOT NULL
    GROUP BY TRIM(company_name)
    ORDER BY jumlah_lowongan DESC
    LIMIT 10
""")
q3.show(truncate=50)

# Q4: Distribusi Waktu Posting
print(" Q4: Distribusi Waktu Posting")

q4 = spark.sql("""
    SELECT
        CASE
            WHEN posted_days_ago IS NULL THEN 'Tidak Diketahui'
            WHEN posted_days_ago <= 1   THEN '≤ 1 hari'
            WHEN posted_days_ago <= 7   THEN '2–7 hari'
            WHEN posted_days_ago <= 30  THEN '8–30 hari'
            ELSE '> 30 hari'
        END AS rentang_posting,
        COUNT(*) AS jumlah_lowongan,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM job_market), 2) AS persentase
    FROM job_market
    GROUP BY rentang_posting
    ORDER BY jumlah_lowongan DESC
""")
q4.show(truncate=50)

# Q5: Rata-rata Salary per Kota (Top 10, Exclude Null Salary)
print(" Q5: Rata-rata Salary per Kota (Top 10)")

q5 = spark.sql("""
    SELECT
        TRIM(location)              AS kota,
        COUNT(*)                    AS jumlah_data,
        ROUND(AVG(salary_numeric), 0) AS rata_rata_salary,
        ROUND(MIN(salary_numeric), 0) AS salary_minimum,
        ROUND(MAX(salary_numeric), 0) AS salary_maksimum
    FROM job_market
    WHERE salary_numeric IS NOT NULL
      AND location IS NOT NULL
    GROUP BY TRIM(location)
    ORDER BY rata_rata_salary DESC
    LIMIT 10
""")
q5.show(truncate=50)

# Simpan hasil query untuk digunakan di Cell 6 (visualisasi)
df_q1 = q1
df_q2 = q2
df_q3 = q3
df_q4 = q4
df_q5 = q5

print("\n Semua query EDA selesai dijalankan!")

 Temp view 'job_market' berhasil dibuat
   Total baris dalam view: 150

 Q1: Top 10 Kota dengan Lowongan Terbanyak
+----------------+---------------+
|            kota|jumlah_lowongan|
+----------------+---------------+
|    Jakarta Raya|             45|
| Jakarta Selatan|             20|
|       Indonesia|             20|
|   Jakarta Utara|             10|
|       Kemayoran|             10|
|Cikarang Selatan|              5|
|  Kebayoran Lama|              5|
|          Malang|              5|
|        Semarang|              5|
|           Krian|              5|
+----------------+---------------+

 Q2: Distribusi Job Type
+--------------+------+----------+
|tipe_pekerjaan|jumlah|persentase|
+--------------+------+----------+
|     Full time|   115|     76.67|
|       Kontrak|    30|     20.00|
|     Freelance|     5|      3.33|
+--------------+------+----------+

 Q3: Top 10 Perusahaan dengan Lowongan Terbanyak
+---------------------------------+---------------+
|                     

In [10]:
# Visualization
# Membuat visualisasi dari hasil EDA menggunakan matplotlib dan seaborn.
# Data Spark dikonversi ke pandas (.toPandas()) HANYA untuk keperluan plotting.
#
# Visualisasi yang dibuat:
#   1. Horizontal bar chart : Top 10 lokasi kerja          → chart_lokasi.png
#   2. Pie chart            : Distribusi job_type          → chart_jobtype.png
#   3. Horizontal bar chart : Top 10 perusahaan            → chart_perusahaan.png
#   4. Bar chart            : Distribusi waktu posting     → chart_posting.png
#   5. WordCloud            : Dari kolom job_requirements  → wordcloud_requirements.png

import matplotlib
matplotlib.use('Agg')  # Non-interactive backend untuk Colab
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
from wordcloud import WordCloud, STOPWORDS
from pyspark.sql import functions as F

# Konfigurasi global styling
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.facecolor" : "white",
    "axes.facecolor"   : "white",
    "axes.edgecolor"   : "#cccccc",
    "axes.labelcolor"  : "#333333",
    "xtick.color"      : "#333333",
    "ytick.color"      : "#333333",
    "text.color"       : "#333333",
    "grid.color"       : "#eeeeee",
    "font.family"      : "DejaVu Sans",
})

PALETTE_BAR  = sns.color_palette("Blues_r", 10)
PALETTE_PIE  = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63", "#9C27B0", "#00BCD4"]
ACCENT_COLOR = "#2196F3"

def save_fig(filename, dpi=150):
    plt.tight_layout()
    plt.savefig(filename, dpi=dpi, bbox_inches="tight",
                facecolor="white")
    plt.show()
    print(f"    Disimpan: {filename}")

print(" Memulai pembuatan visualisasi...\n")

# VIZ 1: Top 10 Lokasi Kerja (Horizontal Bar Chart)
print("[1/5] Membuat chart lokasi kerja...")
pd_q1 = df_q1.toPandas().sort_values("jumlah_lowongan")

fig, ax = plt.subplots(figsize=(12, 6), facecolor="white")
bars = ax.barh(pd_q1["kota"], pd_q1["jumlah_lowongan"],
               color=PALETTE_BAR, edgecolor="none", height=0.65)

# Tambahkan label nilai di ujung bar
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.2, bar.get_y() + bar.get_height() / 2,
            f"{int(w):,}", va="center", ha="left",
            color="#e0e0e0", fontsize=10, fontweight="bold")

ax.set_title("Top 10 Kota dengan Lowongan Terbanyak\n(JobStreet Indonesia)",
             fontsize=15, fontweight="bold", color="white", pad=15)
ax.set_xlabel("Jumlah Lowongan", fontsize=12)
ax.set_ylabel("Kota", fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlim(0, pd_q1["jumlah_lowongan"].max() * 1.2)
save_fig("chart_lokasi.png")

# VIZ 2: Distribusi Job Type (Pie Chart)
print("[2/5] Membuat pie chart job_type...")
pd_q2 = df_q2.filter(F.col("tipe_pekerjaan") != "Tidak Diketahui").toPandas()

pd_q2 = pd_q2[pd_q2["jumlah"] > 0]

fig, ax = plt.subplots(figsize=(10, 7), facecolor="white")
wedges, texts, autotexts = ax.pie(
    pd_q2["jumlah"],
    labels=pd_q2["tipe_pekerjaan"],
    autopct="%1.1f%%",
    startangle=140,
    colors=PALETTE_PIE[:len(pd_q2)],
    pctdistance=0.82,
    wedgeprops=dict(width=0.6, edgecolor="white", linewidth=2),
    textprops=dict(color="white", fontsize=11)
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight("bold")
    at.set_color("white")

ax.set_title("Distribusi Tipe Pekerjaan (Job Type)\n(JobStreet Indonesia)",
             fontsize=15, fontweight="bold", color="white", pad=20)

# Legend
legend_labels = [f"{r['tipe_pekerjaan']} ({r['jumlah']} lowongan)"
                 for _, r in pd_q2.iterrows()]
ax.legend(legend_labels, loc="lower center", bbox_to_anchor=(0.5, -0.12),
          ncol=2, frameon=False, fontsize=10, labelcolor="black")
save_fig("chart_jobtype.png")

# VIZ 3: Top 10 Perusahaan (Horizontal Bar Chart)
print("[3/5] Membuat chart perusahaan...")
pd_q3 = df_q3.toPandas().sort_values("jumlah_lowongan")

fig, ax = plt.subplots(figsize=(13, 6), facecolor="white")
palette_comp = sns.color_palette("rocket", len(pd_q3))
bars = ax.barh(pd_q3["perusahaan"], pd_q3["jumlah_lowongan"],
               color=palette_comp, edgecolor="none", height=0.65)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.05, bar.get_y() + bar.get_height() / 2,
            f"{int(w):,}", va="center", ha="left",
            color="#e0e0e0", fontsize=10, fontweight="bold")

ax.set_title("Top 10 Perusahaan dengan Lowongan Terbanyak\n(JobStreet Indonesia)",
             fontsize=15, fontweight="bold", color="white", pad=15)
ax.set_xlabel("Jumlah Lowongan", fontsize=12)
ax.set_ylabel("Perusahaan", fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlim(0, pd_q3["jumlah_lowongan"].max() * 1.25)
save_fig("chart_perusahaan.png")

# VIZ 4: Distribusi Waktu Posting (Bar Chart)
print("[4/5] Membuat chart distribusi waktu posting...")
pd_q4 = df_q4.filter(F.col("rentang_posting") != "Tidak Diketahui").toPandas()

# Urutkan kategori secara logis
order_map = {"≤ 1 hari": 0, "2–7 hari": 1, "8–30 hari": 2, "> 30 hari": 3}
pd_q4["sort_key"] = pd_q4["rentang_posting"].map(order_map).fillna(99)
pd_q4 = pd_q4.sort_values("sort_key")

fig, ax = plt.subplots(figsize=(10, 6), facecolor="white")
colors_post = ["#e94560", "#e67e22", "#3498db", "#2ecc71"]
bars = ax.bar(pd_q4["rentang_posting"], pd_q4["jumlah_lowongan"],
              color=colors_post[:len(pd_q4)], edgecolor="none",
              width=0.55, zorder=3)

ax.grid(axis="y", alpha=0.3, zorder=0)
for bar in bars:
    h = bar.get_height()
    pct = h / pd_q4["jumlah_lowongan"].sum() * 100
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.3,
            f"{int(h)}\n({pct:.1f}%)",
            ha="center", va="bottom",
            color="white", fontsize=11, fontweight="bold")

ax.set_title("Distribusi Waktu Posting Lowongan\n(JobStreet Indonesia)",
             fontsize=15, fontweight="bold", color="white", pad=15)
ax.set_xlabel("Rentang Waktu Posting", fontsize=12)
ax.set_ylabel("Jumlah Lowongan", fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_ylim(0, pd_q4["jumlah_lowongan"].max() * 1.25)
save_fig("chart_posting.png")

# VIZ 5: WordCloud dari job_requirements
print("[5/5] Membuat WordCloud dari job_requirements...")

# Kumpulkan semua teks job_requirements yang tidak null
req_texts = (
    df_clean
    .filter(F.col("job_requirements").isNotNull())
    .select("job_requirements")
    .rdd
    .map(lambda r: r[0])
    .collect()
)
all_requirements_text = " ".join(req_texts)

# Stopwords gabungan (Inggris + Indonesia)
indonesian_stopwords = {
    "yang", "dan", "di", "ke", "dari", "untuk", "dengan", "atau",
    "pada", "ini", "itu", "adalah", "akan", "dalam", "tidak", "dapat",
    "oleh", "juga", "sebagai", "telah", "ada", "harus", "lebih",
    "serta", "kami", "anda", "tim", "kerja", "pengalaman", "minimal",
    "memiliki", "mampu", "bekerja", "baik", "sangat", "perusahaan",
    "sesuai", "bidang", "nilai", "tambah", "diutamakan", "menjadi",
    "pekerjaan", "posisi", "kandidat", "pendidikan", "jurusan",
    "kemampuan", "seorang", "secara", "antara", "atas", "bawah",
    "para", "tahun", "hari", "setiap", "berbagai", "satu", "dua",
    "tiga", "membantu", "melakukan", "memastikan", "membuat",
    "mengelola", "menggunakan", "memahami", "terkait", "terbuka",
    "fresh", "graduate", "apply", "s1", "min", "minimum"
}
all_stopwords = STOPWORDS.union(indonesian_stopwords)

# Buat WordCloud
wc = WordCloud(
    width=1200, height=600,
    background_color="white",
    stopwords=all_stopwords,
    max_words=120,
    colormap="plasma",
    prefer_horizontal=0.85,
    relative_scaling=0.5,
    min_font_size=10,
    collocations=False
).generate(all_requirements_text)

fig, ax = plt.subplots(figsize=(15, 7.5), facecolor="white")
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Word Cloud: Kata Kunci dalam Job Requirements\n(JobStreet Indonesia — Data Science Jobs)",
             fontsize=16, fontweight="bold", color="white", pad=20)
save_fig("wordcloud_requirements.png", dpi=120)

print("\n Semua visualisasi berhasil dibuat dan disimpan!")
print("    chart_lokasi.png")
print("    chart_jobtype.png")
print("    chart_perusahaan.png")
print("    chart_posting.png")
print("    wordcloud_requirements.png")

 Memulai pembuatan visualisasi...

[1/5] Membuat chart lokasi kerja...
    Disimpan: chart_lokasi.png
[2/5] Membuat pie chart job_type...
    Disimpan: chart_jobtype.png
[3/5] Membuat chart perusahaan...
    Disimpan: chart_perusahaan.png
[4/5] Membuat chart distribusi waktu posting...
    Disimpan: chart_posting.png
[5/5] Membuat WordCloud dari job_requirements...
    Disimpan: wordcloud_requirements.png

 Semua visualisasi berhasil dibuat dan disimpan!
    chart_lokasi.png
    chart_jobtype.png
    chart_perusahaan.png
    chart_posting.png
    wordcloud_requirements.png


In [12]:
# MLlib: K-Means Clustering

# Melakukan klasterisasi lowongan kerja berdasarkan kemiripan teks pada kolom
# job_requirements menggunakan PySpark MLlib (TANPA scikit-learn).
#
# Pipeline NLP:
#   Tokenizer → StopWordsRemover → HashingTF → IDF → KMeans
#
# Parameter:
#   - k        = 4   (jumlah cluster)
#   - seed     = 42  (reproduksibilitas)
#   - maxIter  = 20
#   - numFeatures (HashingTF) = 1000
#
# Output:
#   - Cluster label per baris
#   - Silhouette Score
#   - clustered_dataset_jobstreet.csv

from pyspark.sql import functions as F
from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover, HashingTF, IDF
)
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline

print("Memulai pipeline K-Means Clustering...\n")

# LANGKAH 1: Filter data yang memiliki job_requirements
print("[1/7] Memfilter baris dengan job_requirements tidak null...")
df_ml = df_clean.filter(
    F.col("job_requirements").isNotNull() &
    (F.length(F.trim(F.col("job_requirements"))) > 10)
)
print(f"    Baris tersedia untuk clustering: {df_ml.count():,}")

# LANGKAH 2: Tokenizer — split teks menjadi array kata
print("[2/7] Inisialisasi Tokenizer...")
tokenizer = Tokenizer(
    inputCol="job_requirements",
    outputCol="tokens_raw"
)

# LANGKAH 3: StopWordsRemover — hapus kata-kata umum
print("[3/7] Inisialisasi StopWordsRemover...")
english_stopwords = StopWordsRemover.loadDefaultStopWords("english")
indonesian_sw = [
    "yang", "dan", "di", "ke", "dari", "untuk", "dengan", "atau",
    "pada", "ini", "itu", "adalah", "akan", "dalam", "tidak", "dapat",
    "oleh", "juga", "sebagai", "telah", "ada", "harus", "lebih",
    "serta", "kami", "anda", "tim", "kerja", "pengalaman", "minimal",
    "memiliki", "mampu", "bekerja", "baik", "sangat", "perusahaan",
    "sesuai", "bidang", "nilai", "tambah", "diutamakan", "menjadi",
    "pekerjaan", "posisi", "kandidat", "pendidikan", "jurusan",
    "kemampuan", "seorang", "secara", "antara", "atas", "bawah",
    "membantu", "melakukan", "memastikan", "membuat", "mengelola",
    "menggunakan", "memahami", "terkait", "s1", "d3", "min", "minimum",
    "fresh", "graduate", "tahun", "year", "least", "related", "field",
    "experience", "skills", "ability", "good", "strong", "required",
    "bachelor", "degree", "must", "have", "knowledge", "work", "team"
]
all_stopwords_ml = list(set(english_stopwords + indonesian_sw))

remover = StopWordsRemover(
    inputCol="tokens_raw",
    outputCol="tokens_clean",
    stopWords=all_stopwords_ml
)

# LANGKAH 4: HashingTF — konversi token ke vektor frekuensi
print("[4/7] Inisialisasi HashingTF (numFeatures=1000)...")
hashingTF = HashingTF(
    inputCol="tokens_clean",
    outputCol="raw_features",
    numFeatures=1000
)

# LANGKAH 5: IDF — inverse document frequency weighting
print("[5/7] Inisialisasi IDF...")
idf = IDF(
    inputCol="raw_features",
    outputCol="features",
    minDocFreq=2
)

# LANGKAH 6: KMeans — klasterisasi
print("[6/7] Inisialisasi KMeans (k=4, seed=42, maxIter=20)...")
kmeans = KMeans(
    featuresCol="features",
    predictionCol="cluster",
    k=4,
    seed=42,
    maxIter=20,
    distanceMeasure="euclidean"
)

# Build & Fit Pipeline
print("\nMembangun dan melatih pipeline MLlib...")
pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf, kmeans])
model    = pipeline.fit(df_ml)
df_clustered = model.transform(df_ml)

print("Pipeline selesai dijalankan!")

# LANGKAH 7: Evaluasi & Hasil
print("\n[7/7] Evaluasi hasil clustering...")

# Silhouette Score
evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)
silhouette_score = evaluator.evaluate(df_clustered)
print(f"\nSilhouette Score : {silhouette_score:.4f}")
print("   (Nilai mendekati 1.0 = cluster sangat terpisah baik)")
print(f"   (Nilai mendekati 0.0 = cluster saling tumpang tindih)")

# Distribusi per cluster
print("\nDistribusi Baris per Cluster:")
df_clustered.groupBy("cluster") \
             .count() \
             .orderBy("cluster") \
             .show()

# Tampilkan top 5 baris per cluster
print("\nContoh Lowongan per Cluster (5 baris per cluster):")
for cluster_id in range(4):
    print(f"\n── Cluster {cluster_id} ─────────────────────────")
    df_clustered \
        .filter(F.col("cluster") == cluster_id) \
        .select("job_title", "company_name", "cluster") \
        .show(5, truncate=60)

# Simpan hasil clustering
OUTPUT_CLUSTERED = "clustered_dataset_jobstreet.csv"
print(f"\nMenyimpan hasil clustering ke '{OUTPUT_CLUSTERED}'...")

(
    df_clustered
    .select(
        "job_title", "company_name", "location", "job_type",
        "experience_level", "education_req", "salary_range",
        "salary_numeric", "posted_date", "posted_days_ago",
        "source_platform", "cluster"
    )
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(OUTPUT_CLUSTERED)
)

print(f"Hasil clustering berhasil disimpan ke '{OUTPUT_CLUSTERED}/'")

# Simpan variabel untuk Cell 8
total_clustered = df_clustered.count()
print(f"   Total baris ter-cluster: {total_clustered:,}")

Memulai pipeline K-Means Clustering...

[1/7] Memfilter baris dengan job_requirements tidak null...
    Baris tersedia untuk clustering: 150
[2/7] Inisialisasi Tokenizer...
[3/7] Inisialisasi StopWordsRemover...
[4/7] Inisialisasi HashingTF (numFeatures=1000)...
[5/7] Inisialisasi IDF...
[6/7] Inisialisasi KMeans (k=4, seed=42, maxIter=20)...

Membangun dan melatih pipeline MLlib...
Pipeline selesai dijalankan!

[7/7] Evaluasi hasil clustering...

Silhouette Score : 0.1880
   (Nilai mendekati 1.0 = cluster sangat terpisah baik)
   (Nilai mendekati 0.0 = cluster saling tumpang tindih)

Distribusi Baris per Cluster:
+-------+-----+
|cluster|count|
+-------+-----+
|      0|  135|
|      1|    5|
|      2|    5|
|      3|    5|
+-------+-----+


Contoh Lowongan per Cluster (5 baris per cluster):

── Cluster 0 ─────────────────────────
+-------------------------------------------------------+-------------------------------+-------+
|                                              job_title|  

In [13]:
# Cluster Interpretation & Visualization

# Menginterpretasikan hasil K-Means clustering dengan:
#   1. Mengidentifikasi kata paling sering muncul per cluster (top 20)
#   2. Visualisasi 4 WordCloud dalam grid 2x2 (satu per cluster)
#   3. Tabel ringkasan: cluster id | row count | top 5 keywords |
#                        sample job titles
#
# CATATAN: Konversi ke pandas dilakukan di sini HANYA untuk plotting.
# Semua agregasi (groupBy, kata paling frequent) menggunakan PySpark.

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS
from collections import Counter
from pyspark.sql import functions as F

print("🔍 Menginterpretasikan hasil clustering...\n")

# Stopwords
STOPWORDS_ALL = STOPWORDS.union({
    "yang", "dan", "di", "ke", "dari", "untuk", "dengan", "atau",
    "pada", "ini", "itu", "adalah", "akan", "dalam", "tidak", "dapat",
    "oleh", "juga", "sebagai", "telah", "ada", "harus", "lebih",
    "serta", "kami", "anda", "memiliki", "mampu", "bekerja", "baik",
    "pengalaman", "minimal", "sesuai", "bidang", "nilai", "tambah",
    "diutamakan", "menjadi", "pekerjaan", "posisi", "kandidat",
    "pendidikan", "jurusan", "kemampuan", "seorang", "membantu",
    "melakukan", "memastikan", "membuat", "mengelola", "menggunakan",
    "memahami", "terkait", "s1", "d3", "min", "fresh", "graduate",
    "tahun", "year", "related", "field", "experience", "skills",
    "ability", "good", "strong", "required", "bachelor", "degree",
    "must", "have", "knowledge", "work", "team", "least", "minimum",
    "perusahaan", "sangat", "secara", "terkait", "kerja"
})

# LANGKAH 1: Kumpulkan top words & sample job titles per cluster
print("Mengumpulkan kata-kata teratas per cluster...")

cluster_info = {}  # cluster_id → {words: Counter, titles: list, count: int}

for cid in range(4):
    df_c = df_clustered.filter(F.col("cluster") == cid)
    row_count = df_c.count()

    # Ambil semua teks job_requirements untuk cluster ini
    texts = (
        df_c
        .filter(F.col("job_requirements").isNotNull())
        .select("job_requirements")
        .rdd.map(lambda r: r[0])
        .collect()
    )
    full_text = " ".join(texts).lower()

    # Hitung frekuensi kata (exclude stopwords)
    words = [
        w.strip(".,;:()/[]\"'-")
        for w in full_text.split()
        if len(w) > 2 and w.strip(".,;:()/[]\"'-") not in STOPWORDS_ALL
    ]
    word_freq = Counter(words)

    # Ambil sample job titles (non-null, max 3)
    titles = (
        df_c
        .filter(F.col("job_title").isNotNull())
        .select("job_title")
        .rdd.map(lambda r: r[0])
        .take(3)
    )

    cluster_info[cid] = {
        "count"     : row_count,
        "word_freq" : word_freq,
        "top20"     : [w for w, _ in word_freq.most_common(20)],
        "top5"      : [w for w, _ in word_freq.most_common(5)],
        "top3"      : [w for w, _ in word_freq.most_common(3)],
        "titles"    : titles,
        "full_text" : full_text,
    }
    print(f"   Cluster {cid}: {row_count} baris | Top 5: {cluster_info[cid]['top5']}")

# LANGKAH 2: Visualisasi 4 WordCloud (2x2 grid)
print("\n Membuat 4 WordCloud (grid 2×2)...")

CLUSTER_COLORS = ["plasma", "magma", "viridis", "inferno"]

fig, axes = plt.subplots(2, 2, figsize=(18, 12),
                         facecolor="#0d0d1a")
fig.suptitle(
    "Cluster Interpretation — Word Clouds per K-Means Cluster\n"
    "(JobStreet Indonesia: Data Science Job Requirements)",
    fontsize=18, fontweight="bold", color="white", y=1.01
)

for idx, (ax, cid) in enumerate(zip(axes.flatten(), range(4))):
    info = cluster_info[cid]
    top3_str = ", ".join(info["top3"]) if info["top3"] else "(tidak ada data)"

    if info["full_text"].strip():
        wc = WordCloud(
            width=700, height=400,
            background_color="#0d0d1a",
            stopwords=STOPWORDS_ALL,
            max_words=60,
            colormap=CLUSTER_COLORS[cid],
            prefer_horizontal=0.80,
            relative_scaling=0.5,
            min_font_size=8,
            collocations=False
        ).generate(info["full_text"])
        ax.imshow(wc, interpolation="bilinear")
    else:
        ax.text(0.5, 0.5, "Tidak ada data",
                ha="center", va="center",
                color="white", fontsize=14,
                transform=ax.transAxes)

    ax.axis("off")
    ax.set_facecolor("#0d0d1a")
    ax.set_title(
        f"Cluster {cid}: {top3_str}\n({info['count']} lowongan)",
        color="white", fontsize=13, fontweight="bold", pad=10
    )

plt.tight_layout()
plt.savefig("cluster_wordclouds.png", dpi=130,
            bbox_inches="tight", facecolor="#0d0d1a")
plt.show()
print("    Disimpan: cluster_wordclouds.png")

# LANGKAH 3: Tabel Ringkasan Cluster
print()
print(" TABEL RINGKASAN HASIL CLUSTERING")
print()
print(f"{'Cluster':<10} {'Jumlah':>8} {'Top 5 Keywords':<45} {'Sample Job Titles'}")

for cid in range(4):
    info = cluster_info[cid]
    top5_str   = ", ".join(info["top5"][:5]) if info["top5"] else "-"
    titles_str = " | ".join([t[:30] for t in info["titles"]]) if info["titles"] else "-"

    print(f"Cluster {cid:<3} {info['count']:>8}   {top5_str:<45}")
    print(f"{'':>19} Sample: {titles_str}")
    print("-" * 80)

print("\nInterpretasi Cluster:")
for cid in range(4):
    info = cluster_info[cid]
    top_words = ", ".join(info["top5"]) if info["top5"] else "(tidak ada)"
    print(f"   🔹 Cluster {cid} ({info['count']} lowongan): Fokus pada → {top_words}")

print("\nInterpretasi cluster selesai!")

🔍 Menginterpretasikan hasil clustering...

Mengumpulkan kata-kata teratas per cluster...
   Cluster 0: 135 baris | Top 5: ['data', 'business', 'support', 'requirements', 'system']
   Cluster 1: 5 baris | Top 5: ['artificial', 'intelligence', 'ai', 'tools', 'automasi']
   Cluster 2: 5 baris | Top 5: ['games', 'product', 'new', 'teams', 'ideas']
   Cluster 3: 5 baris | Top 5: ['will', 'quality', 'role', 'help', 'job']

 Membuat 4 WordCloud (grid 2×2)...
    Disimpan: cluster_wordclouds.png

 TABEL RINGKASAN HASIL CLUSTERING

Cluster      Jumlah Top 5 Keywords                                Sample Job Titles
Cluster 0        135   data, business, support, requirements, system
                    Sample: STEM Specialist - Science, Eng | Data Entry | IT & Business System Staff
--------------------------------------------------------------------------------
Cluster 1          5   artificial, intelligence, ai, tools, automasi
                    Sample: Artificial Intelligence (AI) S | Artifi

In [15]:
# Pipeline Summary & Cleanup
# Cell terakhir pipeline ini:
#   1. Mencetak ringkasan lengkap seluruh pipeline (statistik data,
#      model performance, daftar file output)
#   2. Menghentikan SparkSession untuk membebaskan resources

import os

print("  RINGKASAN PIPELINE — Analitik Big Data Final Project")
print("      JobStreet Indonesia | Data Science Job Market Analysis")
print()

# Statistik Data
print("\n STATISTIK DATA:")
print(f"   Total data awal (raw)          : {total_raw:>8,} baris")
print(f"   Total data setelah cleaning    : {total_clean:>8,} baris")
print(f"   Total data untuk clustering    : {total_clustered:>8,} baris")
print(f"   Kolom asal                     : 11 kolom")
print(f"   Kolom setelah cleaning         : {len(df_clean.columns)} kolom")
print(f"   Kolom baru yang ditambahkan    : salary_numeric, posted_days_ago")

# Statistik Clustering
print("\n STATISTIK K-MEANS CLUSTERING:")
print(f"   Algoritma                      : K-Means (PySpark MLlib)")
print(f"   Jumlah cluster (k)             : 4")
print(f"   Fitur                          : TF-IDF dari job_requirements")
print(f"   numFeatures (HashingTF)        : 1000")
print(f"   maxIter                        : 20")
print(f"   Seed                           : 42")
print(f"   Silhouette Score               : {silhouette_score:.4f}")

# Distribusi cluster
print("\n   Distribusi per Cluster:")
for cid in range(4):
    cnt = cluster_info[cid]["count"]
    pct = cnt / total_clustered * 100 if total_clustered > 0 else 0
    top5 = ", ".join(cluster_info[cid]["top5"])
    bar = "█" * int(pct / 2)
    print(f"     Cluster {cid}: {cnt:>3} ({pct:.1f}%) {bar}")
    print(f"              Keywords: {top5}")

# File Output
print("\n FILE OUTPUT YANG DIHASILKAN:")
output_files = [
    ("cleaned_dataset_jobstreet.csv/",   "Dataset setelah cleaning"),
    ("clustered_dataset_jobstreet.csv/", "Dataset hasil K-Means clustering"),
    ("chart_lokasi.png",                 "Chart top 10 kota"),
    ("chart_jobtype.png",                "Pie chart distribusi job_type"),
    ("chart_perusahaan.png",             "Chart top 10 perusahaan"),
    ("chart_posting.png",               "Chart distribusi waktu posting"),
    ("wordcloud_requirements.png",       "WordCloud job_requirements"),
    ("cluster_wordclouds.png",           "WordCloud per cluster (2x2 grid)"),
]

for fname, desc in output_files:
    # Cek keberadaan file (untuk file/folder output)
    base = fname.rstrip("/")
    exists = os.path.exists(base) or os.path.exists(fname)
    status = "Exists" if exists else "Not Exists "
    print(f"   {status} {fname:<42} — {desc}")

# ─── Pipeline Steps Summary ────────────────────────────────────────────────
print("\n TAHAPAN PIPELINE:")
steps = [
    ("Cell 1", "Environment Setup",              "Selesai"),
    ("Cell 2", "SparkSession Initialization",    "Selesai"),
    ("Cell 3", "Data Ingestion & Schema",        "Selesai"),
    ("Cell 4", "Data Cleaning & Preprocessing",  "Selesai"),
    ("Cell 5", "EDA via Spark SQL",              "Selesai"),
    ("Cell 6", "Visualization",                  "Selesai"),
    ("Cell 7", "MLlib K-Means Clustering",       "Selesai"),
    ("Cell 8", "Cluster Interpretation",         "Selesai"),
    ("Cell 9", "Summary & Cleanup",              "Running"),
]
for cell, name, status in steps:
    print(f"   {cell:<8} {name:<38} {status}")

# Tech Stack
print("\n  TECHNOLOGY STACK:")
print(f"   • Apache Spark (PySpark) {spark.version}")
print(f"   • PySpark MLlib   (K-Means, TF-IDF Pipeline)")
print(f"   • Spark SQL       (EDA queries)")
print(f"   • matplotlib + seaborn  (visualisasi)")
print(f"   • wordcloud       (word cloud)")
print(f"   • Dataset         : JobStreet Indonesia")
print(f"   • Mata Kuliah     : Analitik Big Data — Universitas Brawijaya")

print()

# Stop SparkSession
print("\n Menghentikan SparkSession...")
spark.stop()
print("   SparkSession dihentikan.")

print()
print("Pipeline analitik selesai.")

  RINGKASAN PIPELINE — Analitik Big Data Final Project
      JobStreet Indonesia | Data Science Job Market Analysis


 STATISTIK DATA:
   Total data awal (raw)          :      150 baris
   Total data setelah cleaning    :      150 baris
   Total data untuk clustering    :      150 baris
   Kolom asal                     : 11 kolom
   Kolom setelah cleaning         : 13 kolom
   Kolom baru yang ditambahkan    : salary_numeric, posted_days_ago

 STATISTIK K-MEANS CLUSTERING:
   Algoritma                      : K-Means (PySpark MLlib)
   Jumlah cluster (k)             : 4
   Fitur                          : TF-IDF dari job_requirements
   numFeatures (HashingTF)        : 1000
   maxIter                        : 20
   Seed                           : 42
   Silhouette Score               : 0.1880

   Distribusi per Cluster:
     Cluster 0: 135 (90.0%) █████████████████████████████████████████████
              Keywords: data, business, support, requirements, system
     Cluster 1:   5 (3.3%